# ResNet18 feature-space clustering probe

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wxchew/xray_project/blob/main/05_resnet18_feature_clusters.ipynb)

The presentation decks score the trained ResNet18 but say nothing about **what it has learned to
represent**. The HiResCAM panel cannot help either: its probed layer `layer4[-1]` has a 435 px
receptive field on a 224 px image, so position in that map carries no localisation meaning.

This notebook probes the representation instead. Every test image goes through the model, and each
one is summarised by the 512-number vector that comes out of `avgpool` and feeds the final `fc`
layer. That vector is the model's own description of the image. We then reduce those vectors to two
dimensions and cluster them, to ask three things:

1. Does the feature space separate pneumonia from normal on its own, without being told the labels?
2. Is there structure that tracks something **other** than the diagnosis, such as image brightness,
   view, or near-duplicate images?
3. Do the model's 70 mistakes form a coherent group, or are they scattered?

**Pneumonia cases are marked in every scatter plot**, by both colour and marker shape, so the
labelling survives a greyscale print.

Data and the trained checkpoint both live in this repository, so a single shallow clone gives Colab
everything it needs. The architecture, the transform and the dataset loader are imported from
`resnet18_common.py`, which mirrors `04b_chest_Xray_ResNet18.ipynb`, the notebook that trained this
model. This notebook does not restate them.

In [ ]:
# --- Colab setup. No-op when run locally. Safe to re-run. ---
import importlib, os, sys, pathlib

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/wxchew/xray_project.git"

if IN_COLAB:
    # Absolute path on purpose. Using a relative one would clone a nested copy on a
    # re-run, because by then the working directory is already inside the repository.
    REPO_DIR = pathlib.Path("/content/xray_project")
    if (REPO_DIR / ".git").exists():
        # Always refresh. A Colab runtime can hold a clone from an earlier session,
        # and silently running stale code is worse than a slow cell. reset --hard is
        # safe here: this clone is a throwaway with no local edits to lose.
        !git -C {REPO_DIR} fetch --depth 1 -q origin main
        !git -C {REPO_DIR} reset --hard -q FETCH_HEAD
    else:
        !git clone --depth 1 -q {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    !pip -q install umap-learn
    !git -C {REPO_DIR} log --oneline -1

# Drop any cached copy of the helper module. Python keeps imported modules in
# sys.modules, so without this a re-run would keep using the code that was pulled
# the FIRST time, no matter what the files on disk now say.
for name in [m for m in sys.modules if m.split(".")[0] == "resnet18_common"]:
    del sys.modules[name]
importlib.invalidate_caches()

print("Running in Colab:", IN_COLAB)
print("Working directory:", os.getcwd())

In [ ]:
# --- Paths, and hard checks that they point at the right files. ---
# This cell is deliberately loud. A silently wrong path here would produce a
# plausible-looking but meaningless figure ten cells later.

import hashlib, json, collections
from pathlib import Path

REPO       = Path.cwd()
DATA_ROOT  = REPO / "data" / "chest_xray_224"
TEST_DIR   = DATA_ROOT / "test"
TRAIN_DIR  = DATA_ROOT / "train"
CHECKPOINT = REPO / "resnet18_pneumonia_model.pt"
METADATA   = REPO / "resnet18_pneumonia_model.json"
OUT_DIR    = REPO / "feature_clusters"
OUT_DIR.mkdir(exist_ok=True)

# Published in resnet18_results.json. Proves we loaded the *right file*, not just
# a file with the right name.
EXPECTED_SHA = "7d4a4c30439293a8392f8abc246ea1e093bd5bc22e98d0bce263b1c134b8b409"
EXPECTED_COUNTS = {"NORMAL": 234, "PNEUMONIA": 390}

# 1. Every path resolves.
for what, path in [("test dir", TEST_DIR), ("train dir", TRAIN_DIR),
                   ("checkpoint", CHECKPOINT), ("metadata", METADATA)]:
    if not path.exists():
        raise FileNotFoundError(
            f"{what} missing at {path}. "
            "On Colab, re-run the setup cell. Locally, check you are in the repository root."
        )

# 2. Class folders and exact image counts.
counts = {}
for cls in ("NORMAL", "PNEUMONIA"):
    folder = TEST_DIR / cls
    if not folder.is_dir():
        raise FileNotFoundError(f"Expected class folder {folder}")
    counts[cls] = len(sorted(folder.glob("*.jpeg")))
if counts != EXPECTED_COUNTS:
    raise ValueError(f"Test set counts are {counts}, expected {EXPECTED_COUNTS}")

# 3. Checkpoint identity.
digest = hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()
if digest != EXPECTED_SHA:
    raise ValueError(
        f"Checkpoint sha256 is {digest[:16]}..., expected {EXPECTED_SHA[:16]}... "
        "This is a different model than the one the presentation deck reports on."
    )

# 4. The model card must describe the model resnet18_common.py builds. load_metadata
#    raises if the architecture, input shape, class order or normalisation disagree,
#    so the preprocessing cannot silently drift from what the model was trained with.
from resnet18_common import IMAGENET_MEAN, IMAGENET_STD, load_metadata

meta = load_metadata(METADATA)

print(f"repo        {REPO}")
print(f"test dir    {TEST_DIR}")
print(f"            {counts['NORMAL']} NORMAL + {counts['PNEUMONIA']} PNEUMONIA "
      f"= {sum(counts.values())} images")
print(f"checkpoint  {CHECKPOINT.name}  sha256 {digest[:16]}...  OK")
print(f"metadata    {METADATA.name}  mean={IMAGENET_MEAN}  std={IMAGENET_STD}")
print(f"outputs     {OUT_DIR}")

In [ ]:
# --- Leakage check by image content, not by filename. ---
# Filenames collide across splits: train/NORMAL/NORMAL_0.jpeg and test/NORMAL/NORMAL_0.jpeg
# both exist and are DIFFERENT images. A name-based check would false-alarm on every test
# image. Hashing the bytes is the only honest version of this test.

by_hash = collections.defaultdict(list)
for split_dir, split in ((TRAIN_DIR, "train"), (TEST_DIR, "test")):
    for cls in ("NORMAL", "PNEUMONIA"):
        for p in sorted((split_dir / cls).glob("*.jpeg")):
            by_hash[hashlib.sha256(p.read_bytes()).hexdigest()].append(f"{split}/{cls}/{p.name}")

dup_groups  = {h: files for h, files in by_hash.items() if len(files) > 1}
cross_split = [f for f in dup_groups.values() if len({x.split("/")[0] for x in f}) > 1]

print(f"{sum(len(v) for v in by_hash.values())} files, {len(by_hash)} unique images")
print(f"{len(dup_groups)} duplicate groups, {len(cross_split)} of them straddling train and test")

if cross_split:
    print("\nWARNING: test images also appear in train. The reported test score is inflated.")
    for group in cross_split[:20]:
        print("   ", group)
else:
    print("\nNo test image appears anywhere in train. The test evaluation is free of "
          "byte-level duplicate leakage.")

# The duplicates that DO exist sit inside train/. The model card warns they straddle the
# train/valid split used for early stopping. Quantify that, since it is the one real leak.
train_files = set(meta["data"]["train_files"])
valid_files = set(meta["data"]["valid_files"])
straddling = [
    files for files in dup_groups.values()
    if all(x.startswith("train/") for x in files)
    and any(x[len("train/"):] in train_files for x in files)
    and any(x[len("train/"):] in valid_files for x in files)
]
print(f"\n{len(straddling)} duplicate groups straddle the train/valid split. This is the leak "
      "the model card describes. It affects early stopping, not the test score.")

In [ ]:
# --- Load the model. ---
# The architecture, the transform and the loader all live in resnet18_common.py, which
# mirrors 04b_chest_Xray_ResNet18.ipynb, the notebook that trained this checkpoint.
# Restating them here would mean two copies that can drift apart.
# The one thing analysis needs and training does not is features_and_logits(), which
# returns the 512-d avgpool vector alongside the logit from a single forward pass.

import torch
from torch.utils.data import DataLoader

import resnet18_common
from resnet18_common import features_and_logits, load_trained_model, make_dataset

# Print which file was actually imported. If a stale copy is cached from an earlier
# session this line says so at once, instead of surfacing later as a puzzling error.
print("helpers:", resnet18_common.__file__)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

model = load_trained_model(CHECKPOINT, device=DEVICE)
print(f"loaded {sum(p.numel() for p in model.parameters()):,} parameters "
      f"({sum(p.numel() for p in model.net.fc.parameters())} of them in the head)")

In [ ]:
# --- Push all 624 test images through the model and keep the 512-d vectors. ---

import numpy as np

dataset = make_dataset(TEST_DIR)     # same preprocessing the model was trained with

feature_batches, logit_batches, label_batches = [], [], []
with torch.no_grad():
    for images, labels in DataLoader(dataset, batch_size=64, shuffle=False):
        feats, logits = features_and_logits(model, images.to(DEVICE))
        feature_batches.append(feats.cpu().numpy())
        logit_batches.append(logits.squeeze(1).cpu().numpy())
        label_batches.append(labels.numpy())

features      = np.concatenate(feature_batches).astype(np.float32)   # (624, 512)
logits        = np.concatenate(logit_batches).astype(np.float64)
true_labels   = np.concatenate(label_batches).astype(np.int64)
probabilities = 1.0 / (1.0 + np.exp(-logits))
paths         = np.array([p for p, _ in dataset.samples])

assert features.shape == (624, 512), features.shape
assert np.isfinite(features).all(), "non-finite features"
assert (features >= 0).all(), "avgpool output should be non-negative after ReLU"

np.savez_compressed(
    OUT_DIR / "resnet18_features.npz",
    features=features, probabilities=probabilities,
    true_labels=true_labels, paths=paths,
)
print(f"features {features.shape}, saved to {OUT_DIR / 'resnet18_features.npz'}")
print(f"pneumonia {int((true_labels == 1).sum())}   normal {int((true_labels == 0).sum())}")

In [ ]:
# --- Guard: do these features come from the model the deck reports on? ---
# Compare the recomputed scores against the published confusion matrix and AUC.
# The tolerance is deliberate. Colab GPU and local CPU arithmetic differ in the last bits,
# which can flip an image sitting exactly on the 0.5 boundary. Anything wider than one
# image means the wrong checkpoint or the wrong preprocessing, and we should stop.

from sklearn.metrics import confusion_matrix, roc_auc_score

PUBLISHED_CM  = np.array([[168, 66], [4, 386]])   # rows are true [NORMAL, PNEUMONIA]
PUBLISHED_AUC = 0.9844291036598728

predicted = (probabilities > 0.5).astype(int)
cm  = confusion_matrix(true_labels, predicted)
auc = roc_auc_score(true_labels, probabilities)

print("confusion matrix (rows = true NORMAL / PNEUMONIA):")
print(cm)
print(f"\nAUC {auc:.6f}   published {PUBLISHED_AUC:.6f}   gap {abs(auc - PUBLISHED_AUC):.2e}")

if np.abs(cm - PUBLISHED_CM).max() > 1:
    raise ValueError(f"Confusion matrix {cm.tolist()} differs from the published "
                     f"{PUBLISHED_CM.tolist()} by more than one image.")
if abs(auc - PUBLISHED_AUC) > 1e-3:
    raise ValueError(f"AUC {auc:.6f} differs from the published {PUBLISHED_AUC:.6f}")

errors = predicted != true_labels
print(f"\nGuard passed. {int(errors.sum())} misclassified images: "
      f"{int((predicted[true_labels == 0] == 1).sum())} false positives, "
      f"{int((predicted[true_labels == 1] == 0).sum())} false negatives.")

In [ ]:
# --- PCA on the representation. ---
# The avgpool output is non-negative, because it follows a ReLU. So every vector sits in
# one corner of the space and raw distances are dominated by overall magnitude.
# L2-normalising first means we compare *direction*, which is the cosine geometry the
# final linear layer actually uses.

import matplotlib.pyplot as plt

normalised = features / np.linalg.norm(features, axis=1, keepdims=True)
centred    = normalised - normalised.mean(axis=0, keepdims=True)

U, S, Vt  = np.linalg.svd(centred, full_matrices=False)
explained = S**2 / np.sum(S**2)
scores    = U * S                      # (624, 512) coordinates in PC space
N_PC      = 50
pcs       = scores[:, :N_PC]

cumulative = np.cumsum(explained)
n90 = int(np.searchsorted(cumulative, 0.90) + 1)
print(f"PC1 explains {explained[0]:.1%}, PC2 {explained[1]:.1%}, PC3 {explained[2]:.1%}")
print(f"{n90} components reach 90% of the variance. "
      f"The {N_PC} kept here cover {cumulative[N_PC - 1]:.1%}.")

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(range(1, 31), explained[:30] * 100, marker="o", ms=4, lw=1.2, color="#2b6cb0")
ax.set_xlabel("Principal component")
ax.set_ylabel("Variance explained (%)")
ax.set_title("Scree plot of the 512-d representation", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# --- UMAP for the 2-D view, k-means for the clusters. ---
# UMAP runs on the 50 PCs rather than the raw 512 dimensions. It is faster, and it strips
# the low-variance directions that are mostly noise.

import umap
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

reducer   = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="euclidean", random_state=0)
embedding = reducer.fit_transform(pcs)
print("UMAP embedding:", embedding.shape)

print("\n k   silhouette   adjusted Rand vs true label")
sweep = {}
for k in range(2, 11):
    km  = KMeans(n_clusters=k, random_state=0, n_init=10).fit(pcs)
    sil = silhouette_score(pcs, km.labels_)
    ari = adjusted_rand_score(true_labels, km.labels_)
    sweep[k] = (sil, ari, km.labels_)
    print(f"{k:2d}   {sil:10.4f}   {ari:26.4f}")

BEST_K = max(sweep, key=lambda k: sweep[k][0])
best_sil, best_ari, clusters = sweep[BEST_K]
print(f"\nSilhouette picks k = {BEST_K}: silhouette {best_sil:.4f}, adjusted Rand {best_ari:.4f}")
print("Adjusted Rand near 0 means the clusters ignore the diagnosis. Near 1 means they recover it.")

print("\ncluster   size   pneumonia fraction")
for c in range(BEST_K):
    m = clusters == c
    print(f"{c:7d}   {int(m.sum()):4d}   {true_labels[m].mean():18.3f}")

In [ ]:
# --- The four-panel figure. Pneumonia is marked in every scatter. ---
# Colour AND marker shape both encode the label, so the figure still reads in greyscale.

NORMAL_STYLE    = dict(color="#4c9be8", marker="o", s=16, alpha=0.75, linewidths=0)
PNEUMONIA_STYLE = dict(color="#d1495b", marker="^", s=18, alpha=0.75, linewidths=0)

is_pneu = true_labels == 1
n_pneu, n_norm = int(is_pneu.sum()), int((~is_pneu).sum())

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
(axA, axB), (axC, axD) = axes


def label_scatter(ax, xy, title, xlabel, ylabel):
    ax.scatter(xy[~is_pneu, 0], xy[~is_pneu, 1], label=f"NORMAL (n={n_norm})", **NORMAL_STYLE)
    ax.scatter(xy[is_pneu, 0],  xy[is_pneu, 1],  label=f"PNEUMONIA (n={n_pneu})", **PNEUMONIA_STYLE)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend(loc="best", frameon=True, fontsize=9, markerscale=1.6)
    ax.spines[["top", "right"]].set_visible(False)


# A: PCA, coloured by true label
label_scatter(axA, scores[:, :2], "PCA of the 512-d representation",
              f"PC1 ({explained[0]:.1%} of variance)", f"PC2 ({explained[1]:.1%})")

# B: UMAP, coloured by true label
label_scatter(axB, embedding, "UMAP, coloured by true diagnosis", "UMAP 1", "UMAP 2")

# C: UMAP, coloured by k-means cluster.
# Cluster composition goes in the legend, not as labels on the points: two clusters with
# nearby centroids would print their labels on top of each other.
palette = plt.cm.tab10(np.linspace(0, 1, 10))
for c in range(BEST_K):
    m = clusters == c
    axC.scatter(embedding[m, 0], embedding[m, 1], s=16, alpha=0.75, linewidths=0,
                color=palette[c % 10],
                label=f"cluster {c}: n={int(m.sum())}, {true_labels[m].mean():.0%} pneumonia")
axC.set_title(f"UMAP, k-means clusters (k={BEST_K}, adjusted Rand={best_ari:.2f})", fontsize=11)
axC.set_xlabel("UMAP 1")
axC.set_ylabel("UMAP 2")
axC.legend(loc="best", fontsize=8, markerscale=1.4)
axC.spines[["top", "right"]].set_visible(False)

# D: UMAP, coloured by model probability, with every error ringed
sc = axD.scatter(embedding[:, 0], embedding[:, 1], c=probabilities, cmap="coolwarm",
                 vmin=0, vmax=1, s=16, alpha=0.9, linewidths=0)
axD.scatter(embedding[errors, 0], embedding[errors, 1], s=70, facecolors="none",
            edgecolors="black", linewidths=1.0, label=f"misclassified (n={int(errors.sum())})")
fig.colorbar(sc, ax=axD, label="P(pneumonia)", fraction=0.046, pad=0.04)
axD.set_title("UMAP, coloured by model confidence", fontsize=11)
axD.set_xlabel("UMAP 1")
axD.set_ylabel("UMAP 2")
axD.legend(loc="best", fontsize=9)
axD.spines[["top", "right"]].set_visible(False)

for ax, letter in zip([axA, axB, axC, axD], "ABCD"):
    ax.text(-0.08, 1.06, letter, transform=ax.transAxes, fontsize=16, fontweight="bold")

fig.suptitle(f"ResNet18 feature space, test set (n={len(true_labels)})", fontsize=14, y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.965])
fig.savefig(OUT_DIR / "feature_clusters_four_panel.png", dpi=300,
            bbox_inches="tight", facecolor="white")
fig.savefig(OUT_DIR / "feature_clusters_four_panel.pdf",
            bbox_inches="tight", facecolor="white")
plt.show()
print("saved to", OUT_DIR / "feature_clusters_four_panel.pdf")

In [ ]:
# --- Three probes for structure that is NOT the diagnosis. ---

from PIL import Image
from scipy.stats import pearsonr

summary = {}

# [1] Near-duplicates in feature space: two images the model cannot tell apart.
cosine = normalised @ normalised.T
i_idx, j_idx = np.where(np.triu(cosine, k=1) > 0.999)
near_dupes = [
    {"a": str(Path(paths[i]).relative_to(REPO)),
     "b": str(Path(paths[j]).relative_to(REPO)),
     "cosine": round(float(cosine[i, j]), 6),
     "same_label": bool(true_labels[i] == true_labels[j])}
    for i, j in zip(i_idx, j_idx)
]
summary["near_duplicate_pairs"] = near_dupes
print(f"[1] {len(near_dupes)} image pairs with cosine similarity above 0.999")
for d in near_dupes[:10]:
    print(f"    {d['cosine']:.5f}  same label={d['same_label']}  {d['a']}  <->  {d['b']}")
if not near_dupes:
    print("    None. The model gives every test image a distinguishable representation.")

# [2] Do the top components track plain image properties rather than the diagnosis?
# Every image here is already 224x224, so aspect ratio is constant and useless as a probe.
# "border brightness" replaces it: the mean of the outer 10% frame. It is the direct test
# of the worry raised by the HiResCAM panels, that the model might key on the image edge
# rather than the lungs.
stats = []
for p in paths:
    with Image.open(p) as im:
        grey = np.asarray(im.convert("L"), dtype=np.float32)
    edge = max(1, round(0.10 * min(grey.shape)))
    border = np.concatenate([grey[:edge].ravel(), grey[-edge:].ravel(),
                             grey[:, :edge].ravel(), grey[:, -edge:].ravel()])
    stats.append((grey.mean(), grey.std(), border.mean()))
stats = np.array(stats)
confounds = {"mean intensity":    stats[:, 0],
             "intensity sd":      stats[:, 1],
             "border brightness": stats[:, 2]}

# A constant column makes Pearson undefined. Drop it rather than print nan.
constant = [n for n, v in confounds.items() if np.std(v) == 0]
for n in constant:
    print(f"    (skipping '{n}': identical for every image, so no correlation exists)")
    confounds.pop(n)

print("\n[2] Pearson correlation of each PC with plain image properties")
print("    " + "".join(f"{name:>19s}" for name in ["component"] + list(confounds)))
conf_table = {}
for pc in range(5):
    row = {name: round(float(pearsonr(scores[:, pc], v)[0]), 4) for name, v in confounds.items()}
    conf_table[f"PC{pc + 1}"] = row
    print(f"    {'PC' + str(pc + 1):>19s}" + "".join(f"{row[n]:19.3f}" for n in confounds))

label_r = {n: float(pearsonr(v, true_labels)[0]) for n, v in confounds.items()}
print(f"    {'the label itself':>19s}" + "".join(f"{label_r[n]:19.3f}" for n in confounds))
summary["pc_confound_correlation"] = conf_table
summary["label_confound_correlation"] = {k: round(v, 4) for k, v in label_r.items()}

pc1_vs_label = abs(float(pearsonr(scores[:, 0], true_labels)[0]))
worst = max(conf_table["PC1"], key=lambda n: abs(conf_table["PC1"][n]))
print(f"\n    PC1 correlates {abs(conf_table['PC1'][worst]):.3f} with {worst}, "
      f"and {pc1_vs_label:.3f} with the diagnosis.")
print("    If the first number is the larger one, the dominant axis of the representation")
print("    is an image property, not the disease.")
summary["pc1_vs_label_correlation"] = round(pc1_vs_label, 4)

# [3] Where do the mistakes sit?
centroids = {}
for c in (0, 1):
    v = normalised[true_labels == c].mean(axis=0)
    centroids[c] = v / np.linalg.norm(v)

d_true = np.array([1.0 - normalised[i] @ centroids[true_labels[i]] for i in range(len(paths))])

fp = (predicted == 1) & (true_labels == 0)
fn = (predicted == 0) & (true_labels == 1)
print("\n[3] Cosine distance to the true-class centroid. Larger means more atypical.")
print(f"    correct         {d_true[~errors].mean():.4f}   (n={int((~errors).sum())})")
print(f"    false positive  {d_true[fp].mean():.4f}   (n={int(fp.sum())})")
print(f"    false negative  {d_true[fn].mean():.4f}   (n={int(fn.sum())})")

fp_clusters = collections.Counter(clusters[fp].tolist())
print(f"\n    The {int(fp.sum())} false positives land in clusters: "
      + ", ".join(f"{c}:{n}" for c, n in sorted(fp_clusters.items())))
print("    Concentrated in one cluster means a coherent hard subgroup. Spread evenly means noise.")

summary["error_geometry"] = {
    "mean_cosine_distance_to_true_centroid": {
        "correct":        round(float(d_true[~errors].mean()), 4),
        "false_positive": round(float(d_true[fp].mean()), 4),
        "false_negative": round(float(d_true[fn].mean()), 4),
    },
    "false_positive_cluster_counts": {str(c): int(n) for c, n in sorted(fp_clusters.items())},
}
summary["clustering"] = {
    "best_k": int(BEST_K),
    "silhouette": round(float(best_sil), 4),
    "adjusted_rand_vs_label": round(float(best_ari), 4),
    "silhouette_sweep": {str(k): round(float(v[0]), 4) for k, v in sweep.items()},
}
summary["pca"] = {
    "variance_explained_top5": [round(float(x), 4) for x in explained[:5]],
    "components_for_90pct": n90,
}
summary["duplicate_scan"] = {
    "duplicate_groups": len(dup_groups),
    "train_test_duplicates": len(cross_split),
    "train_valid_straddling_groups": len(straddling),
}

In [ ]:
# --- Write everything out, and download it before Colab throws the runtime away. ---

import pandas as pd

table = pd.DataFrame({
    "path": [str(Path(p).relative_to(REPO)) for p in paths],
    "true_label": ["PNEUMONIA" if t else "NORMAL" for t in true_labels],
    "predicted_label": ["PNEUMONIA" if p else "NORMAL" for p in predicted],
    "pneumonia_probability": probabilities,
    "correct": ~errors,
    "cluster": clusters,
    "umap_1": embedding[:, 0],
    "umap_2": embedding[:, 1],
})
for pc in range(5):
    table[f"pc{pc + 1}"] = scores[:, pc]
table.to_csv(OUT_DIR / "cluster_assignments.csv", index=False)

(OUT_DIR / "cluster_summary.json").write_text(json.dumps(summary, indent=2))

print("wrote:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"   {f.name}  ({f.stat().st_size / 1024:.1f} KB)")

if IN_COLAB:
    import shutil
    from google.colab import files
    archive = shutil.make_archive("/content/feature_clusters", "zip", str(OUT_DIR))
    print(f"\nDownloading {archive}. Colab wipes the disk when the runtime disconnects.")
    files.download(archive)

## Reading the output

**Panels A and B** answer the first question. If pneumonia and normal separate into two regions
without the reduction ever seeing a label, the model has learned a representation where the
diagnosis is the dominant structure. Heavy overlap means the separation only appears after the
final linear layer.

**Panel C** answers it more strictly. The adjusted Rand index compares the unsupervised clusters
against the true labels: near 1 means the clusters recovered the diagnosis on their own, near 0
means they found something else entirely. The percentage printed on each cluster says what fraction
of it is pneumonia.

**Panel D** shows where the model is unsure. The ringed points are its mistakes. If they sit
together in one region, there is a specific kind of chest X-ray this model handles badly, and that
is a fixable problem. If they are scattered, the errors are closer to irreducible noise.

**Probe 2 is the one to watch.** If PC1 correlates more strongly with plain image brightness than
with the diagnosis, then the biggest axis of variation in the model's representation is an
acquisition artefact, not disease. That would be a real finding about this dataset, and a caution
for anyone reading the headline accuracy.

### Known limits

- The test set drove model selection historically, so its scores are an upper bound. The duplicate
  scan shows no byte-identical test image in train, but it cannot rule out the same patient
  appearing in both. Patient IDs are not in this dataset.
- UMAP distances *between* separate clumps carry no meaning. Only local neighbourhoods do.
- k-means assumes roughly round, similar-sized clusters. A flat silhouette curve across every k is
  itself an answer: it means there are no clean clusters to find.
- The near-duplicate probe works on the model's features, so it finds images the *model* cannot
  tell apart. That is a wider net than byte-identical files, and a stricter test of redundancy.